## <font color=red>여기까지 싹 다 무시

### MySQL과 python 연동에서의 유의할 점
#### dotenv

- 서버의 정보(주소, 포트, ID, PW, DB)를 코드에 그대로 입력
- 이후 github 같은 곳에 코드를 업로드하면(openkey) 보안상 취약점이 됨
    - 이런 민감한 정보들은 숨겨서 관리

In [1]:
! pip install python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# 설치할 때 사용하는 라이브러리의 이름과 import 할 때의 이름이 다른 경우들이 종종 발생

from dotenv import load_dotenv
import pymysql
import os
import pandas as pd

In [2]:
# os → python 환경에서 사용하는 변수들에 접근하기 위해 사용
# load_dotenv → .env 파일의 내용을 환경 변수에 임시 등록

load_dotenv()

# 등록된 환경변수에 접근
# 가져온다(get) + 환경(env) → getenv(변수명)
# os.getenv('port')
    # 문자열로 출력

True

In [3]:
# DB와 연결

_db = pymysql.connect(
    host = os.getenv('host'),
    port = int(os.getenv('port')),
    user = os.getenv('user'),
    password = os.getenv('pwd'),
    db = os.getenv('db_name')
)

In [4]:
# Cursor 생성
cursor = _db.cursor(pymysql.cursors.DictCursor)

In [5]:
# 테이블 생성
# 기존의 테이블이 존재한다면 테이블 생성 X
# 없으면 생성

create_table = """
    CREATE TABLE IF NOT EXISTS
    `user_info`
    (
        `id` VARCHAR(32) PRIMARY KEY,
        `password` VARCHAR(32) NOT NULL,
        `name` VARCHAR(32),
        `age` INT
    )
"""

# DDL 구문을 생성 → cursor에 질의를 보낸다
# table을 생성하는 과정(DDL)은 확인 절차 없이 DB server에 바로 등록
# cursor를 통해서 sql 쿼리문을 실행할 때는 execute() 함수를 이용
cursor.execute(create_table)

0

In [6]:
# 테이블에 데이터를 대입 → insert 구문

signup_query = """
    INSERT INTO
    `user_info`
    VALUES (
        "test", "1234", "kim", 30
    )
"""

cursor.execute(signup_query)

1

In [ ]:
user_list_query = """
    SELECT
    *
    FROM
    `user_info`
"""

cursor.execute(user_list_query)

1

In [ ]:
#  결과값을 불러오는 함수 → fetchall()
cursor.fetchall()

[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30}]

In [ ]:
# db에서 확정 작업

_db.commit()

In [ ]:
# query문에 들어오는 데이터가 매번 바뀔 때마다 query문을 작성하는 건 불필요한 작업

signup_query = """
    INSERT INTO `user_info`
    VALUES (%s, %s, %s, %s)
"""

# %s 는 들어오는 데이터들의 위치를 지정
input_id = input('아이디를 입력하시오')
input_pass = input('비밀번호를 입력하시오')
input_name = input('이름을 입력하시오')
input_age = input('나이를 입력하시오')

# execute(query, [datas]) 함수에 리스트의 각각의 원소들이 %s 위치에 순서대로 대입
cursor.execute(signup_query, [input_id, input_pass, input_name, input_age])

In [ ]:
cursor.execute(user_list_query)

2

In [ ]:
cursor.fetchall()

[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30},
 {'id': 'test3', 'password': '2222', 'name': 'park', 'age': 10}]

In [ ]:
_db.close()

### DB 연동 class 선언

- 생성자 함수
    - 서버의 정보를 입력한다. (매개변수)
    - class가 생성이 될 때마다 다른 DB server에 정보를 담을 수 있다.
    - 2개의 객체를 생성하여 다른 DB server와의 연동

- 함수 2개 생성
    - query문 실행 함수 (매개변수: query문(필수 입력), *datas(query에 %s가 존재하지 않으면 필요 x))
        - 서버와의 연결
        - cursor 생성
        - CUD (insert, update, delete)
            - query문 작성
            - execute() 함수를 이용하여 cursor에 질의를 보낸다
                - cursor의 테이블에서 데이터의 변화가 생기는 부분
        - R (select)
            - query문 작성
            - execute() 함수를 이용하여 cursor에 질의를 보낸다.
            - cursor에서 데이터를 가져온다.
        - query문의 시작이 select라면 fetchall() 함수를 사용하여 데이터를 return
    - DataBase에 변화를 주는 함수
        - DB server에서 commit()를 이용해서 데이터 확정
        - DB server와의 연결을 종료(close())

In [ ]:
# 여기부터 run

from dotenv import load_dotenv
import pymysql
import os
import pandas as pd

In [ ]:
# class 선언
class MyDB:

    # 생성자 함수: 서버의 정보를 받아오기 위해 사용
    def __init__(self, host, port, user, password, db):
        # 서버의 정보를 인자로 받아와서 객체 내부에서 독립적인 변수에 저장
        self.host = host
        self.port = port
        self.user = user
        self.password = password
        self.db = db


    # DB에 변화를 주는 함수
    def commit(self):
        try:
            # DB에 데이터를 확정(동기화)
            self.db_server.commit()
            # 서버와의 연결을 종료
            self.db_server.close()
            # close() 함수를 사용하더라도 self.db_server의 변수는 사라지지 않는다.
            # 변수 자체를 제거
            del self.db_server
        except:
            # 문제가 발생하는 이유: 서버와의 연결이 되지 않은 경우
            # self.db_server에 데이터가 없거나, 아예 존재하지 않는 경우
            print('데이터베이스 서버와의 연결이 되어있지 않습니다. sql_query() 함수를 호출하여 서버와의 연결을 해주세요')
    
    
    def sql_query(self, query, *datas): 
        # query: sql query문이 입력
        # datas: query문에서 사용될 데이터 목록
        try:
            self.db_server
            print('접속된 서버가 존재함')
        except:
            # DB_server와의 연결
            self.db_server = pymysql.connect(
                host = self.host,
                port = self.port,
                user = self.user,
                password = self.password,
                db = self.db
            )
            
        # cursor 생성
        cursor = self.db_server.cursor(pymysql.cursors.DictCursor)

        
        
        # self.변수명 // 변수명의 차이는?
            # self.변수 → 독립적으로 객체 안에 저장되는 변수
                # (함수 호출 후에도 데이터 존재)
            # 변수 → 함수 호출시 생성이 되고 함수가 종료되면 휘발성으로 사라짐
                # (완벽한 지역변수)
        # CUD의 경우에는 execute() 쿼리문을 cursor에 질의 보낸다.

        try:
            cursor.execute(query, datas)

            # query가 select문이라면?
            # select * from table // SELECT * FROM table → 두 가지의 경우 참
            # 좌측 공백 제거, 소문자 통일, 시작값이 select와 같은가?
            if query.lstrip().lower().startswith('select'):
                result = cursor.fetchall()
            else:
                result = "Query OK!"
            
            return result

        except Exception as e:
            print('query문 execute 중 에러')
            print(e)


In [ ]:
# class 생성

db1= MyDB(
    host = os.getenv('host'),
    port = 3306,
    user = os.getenv('user'),
    password = os.getenv('pwd'),
    db = os.getenv('db_name')
)

In [ ]:
db1.sql_query('select * from `emp`')

OperationalError: (1045, "Access denied for user 'hkssn'@'localhost' (using password: NO)")

In [ ]:
db1.commit()

데이터베이스 서버와의 연결이 되어있지 않습니다. sql_query() 함수를 호출하여 서버와의 연결을 해주세요


##### db2

In [ ]:
db2 = MyDB(
    host = os.getenv('host2'),
    port = int(os.getenv('port2')),
    user = os.getenv('user2'),
    password = os.getenv('pwd2'),
    db = os.getenv('db_name2')
)

In [ ]:
db2.sql_query('select * from `AAPL`')

[{'Date': '1980-12-12',
  'Open': 0.513393,
  'High': 0.515625,
  'Low': 0.513393,
  'Close': 0.513393,
  'Adj Close': 0.410525,
  'Volume': 117258400},
 {'Date': '1980-12-15',
  'Open': 0.488839,
  'High': 0.488839,
  'Low': 0.486607,
  'Close': 0.486607,
  'Adj Close': 0.389106,
  'Volume': 43971200},
 {'Date': '1980-12-16',
  'Open': 0.453125,
  'High': 0.453125,
  'Low': 0.450893,
  'Close': 0.450893,
  'Adj Close': 0.360548,
  'Volume': 26432000},
 {'Date': '1980-12-17',
  'Open': 0.462054,
  'High': 0.464286,
  'Low': 0.462054,
  'Close': 0.462054,
  'Adj Close': 0.369472,
  'Volume': 21610400},
 {'Date': '1980-12-18',
  'Open': 0.475446,
  'High': 0.477679,
  'Low': 0.475446,
  'Close': 0.475446,
  'Adj Close': 0.380182,
  'Volume': 18362400},
 {'Date': '1980-12-19',
  'Open': 0.504464,
  'High': 0.506696,
  'Low': 0.504464,
  'Close': 0.504464,
  'Adj Close': 0.403385,
  'Volume': 12157600},
 {'Date': '1980-12-22',
  'Open': 0.529018,
  'High': 0.53125,
  'Low': 0.529018,
  'Cl

In [ ]:
db2.commit()

##### 다시 db1

In [ ]:
# select 확인 끝, insert/update/delete 확인

insert_query = """
    INSERT INTO `user_info`
    VALUES (%s, %s, %s, %s)
"""

select_query = """
    SELECT * FROM `user_info`
"""

data_list = ['test3', '0000', 'lee', 40]

db1.sql_query(insert_query, *data_list)

OperationalError: (1045, "Access denied for user 'hkssn'@'localhost' (using password: NO)")

##### git 코드

In [ ]:
# class 선언 
class MyDB:
    # 생성자 함수 (서버의 정보를 받아오기 위해 사용)
    def __init__(self, host, port, user, password, db):
        # 서버의 정보를 인자로 받아와서 객체 내부에서 독립적인 변수에 저장
        self.host = host
        self.port = port
        self.user = user
        self.password = password
        self.db = db

    # 데이터베이스에 변화를 주는 함수 
    def commit(self):
        try:
            # DataBase에 데이터를 확정 (동기화)
            self.db_server.commit()
            # 서버와의 연결을 종료 ( 중요한 부분 )
            self.db_server.close()
            # close() 함수를 사용하더라도 self.db_server의 변수는 사라지지 않는다. 
            # 변수 자체를 제거 
            del self.db_server
        except:
            # 문제가 발생하는 이유는? -> 서버와의 연결이 되지 않은 경우 (self.db_server라는 변수에 데이터가 없거나 아예 존재하지 않는 경우)
            print( "데이터베이스 서버와의 연결이 되어있지 않습니다. sql_query() 함수를 호출하여 서버와의 연결을 해주세요" )
    
    def sql_query(self, query, *datas):
        # query : sql query문이 입력이되는 매개변수 
        # datas : query문에서 사용이 될 데이터의 목록

        # 문제점 : 서버의 재접속으로 commit 전의 데이터가 날아감. 
        # 이미 접속중인 경우 재접속 금지 (2026.04.20 update)
        # 해결 방법 self.db_server에 데이터가 존재한다면? -> 서버의 접속중이다. 
        try:
            self.db_server
            # 변수가 존재하지 않으면 NameError 발생 
            print('접속된 서버가 존재함')
        except:
            # 예외 상황이 발생하면 서버와 연결(self.db_server 라는 변수가 없을 시 실행)
            # DB_server와의 연결 
            self.db_server = pymysql.connect(
                host = self.host, 
                port = self.port, 
                user = self.user, 
                password = self.password, 
                db = self.db
            )
        # cursor 생성 
        cursor = self.db_server.cursor(pymysql.cursors.DictCursor)

        # self.변수명 // 변수명의 차이는?
            # self.변수 -> 독립적으로 객체 안에 저장이 되는 변수 ( 함수 호출 후에도 데이터가 존재 )
            # 변수 -> 함수 호출시 생성이 되고 함수가 종료가 되면 휘발성으로 사라짐
        try:
            # CUD의 경우에는 execute() 쿼리문을 커서에 질의 보낸다. R (select도 여기까지는 공통의 작업)
            # execute( query, () ) -> 호출 가능 
            # execute( query, (1,2,3) ) -> 호출 가능 
            cursor.execute(query, datas)
            # query가 select문이라면? 
            # select * from table // SELECT * FROM table, """   select * from table   """ -> 두가지의 경우 모두 참 
            # 좌측 공백을 제거 , 소문자를 통일 , 시작값이 select와 같은가 startwith()
            if query.lstrip().lower().startswith('select'):
                # 결과값을 받아온다. 
                result = cursor.fetchall()
            else:
                result = "Query OK!"
            return result
        except Exception as e:
            print('query문 execute중 에러')
            print(e)

In [ ]:
# class 생성 (서버의 정보를 등록)
db1 = MyDB(
    host = os.getenv('host'), 
    port = 3306, 
    user = os.getenv('user'),
    password = os.getenv('pwd'), 
    db = os.getenv('db_name')
)

In [ ]:
# select 확인이 끝났으니 insert update delete 확인 
insert_query = """
    INSERT INTO `user_info`
    VALUES (%s, %s, %s, %s)
"""
select_query = """
    SELECT * FROM `user_info`
"""

data_list = ['test3', '0000', 'lee', 40]
db1.sql_query(insert_query, *data_list)

In [ ]:
# 서버와의 연결 
db1.sql_query('select * from `emp`')

OperationalError: (1045, "Access denied for user 'hkssn'@'localhost' (using password: NO)")

In [ ]:
db1.sql_query(select_query)

NameError: name 'select_query' is not defined

In [ ]:
update_query = """
    UPDATE `user_info` SET `password` = %s
    WHERE  `id` = %s
"""

data_list = ['0123', 'test3']

db1.sql_query(update_query, *data_list)

OperationalError: (1045, "Access denied for user 'hkssn'@'localhost' (using password: NO)")

## 여기서부터 다시

In [1]:
# 여기부터 run

from dotenv import load_dotenv
import pymysql
import os
import pandas as pd

In [2]:
load_dotenv()

True

In [3]:
_db = pymysql.connect(
    host = os.getenv('host'),
    port = int(os.getenv('port')),
    user = os.getenv('user'),
    password = os.getenv('pwd'),
    db = os.getenv('db_name')
)

In [4]:
cursor = _db.cursor(pymysql.cursors.DictCursor)

In [5]:
# class 선언 
class MyDB:
    # 생성자 함수 (서버의 정보를 받아오기 위해 사용)
    def __init__(self, host, port, user, password, db):
        # 서버의 정보를 인자로 받아와서 객체 내부에서 독립적인 변수에 저장
        self.host = host
        self.port = port
        self.user = user
        self.password = password
        self.db = db

    # 데이터베이스에 변화를 주는 함수 
    def commit(self):
        try:
            # DataBase에 데이터를 확정 (동기화)
            self.db_server.commit()
            # 서버와의 연결을 종료 ( 중요한 부분 )
            self.db_server.close()
            # close() 함수를 사용하더라도 self.db_server의 변수는 사라지지 않는다. 
            # 변수 자체를 제거 
            del self.db_server
        except:
            # 문제가 발생하는 이유는? -> 서버와의 연결이 되지 않은 경우 (self.db_server라는 변수에 데이터가 없거나 아예 존재하지 않는 경우)
            print( "데이터베이스 서버와의 연결이 되어있지 않습니다. sql_query() 함수를 호출하여 서버와의 연결을 해주세요" )
    
    def sql_query(self, query, *datas):
        # query : sql query문이 입력이되는 매개변수 
        # datas : query문에서 사용이 될 데이터의 목록

        # 문제점 : 서버의 재접속으로 commit 전의 데이터가 날아감. 
        # 이미 접속중인 경우 재접속 금지 (2026.04.20 update)
        # 해결 방법 self.db_server에 데이터가 존재한다면? -> 서버의 접속중이다. 
        try:
            self.db_server
            # 변수가 존재하지 않으면 NameError 발생 
            print('접속된 서버가 존재함')
        except:
            # 예외 상황이 발생하면 서버와 연결(self.db_server 라는 변수가 없을 시 실행)
            # DB_server와의 연결 
            self.db_server = pymysql.connect(
                host = self.host, 
                port = self.port, 
                user = self.user, 
                password = self.password, 
                db = self.db
            )
        # cursor 생성 
        cursor = self.db_server.cursor(pymysql.cursors.DictCursor)

        # self.변수명 // 변수명의 차이는?
            # self.변수 -> 독립적으로 객체 안에 저장이 되는 변수 ( 함수 호출 후에도 데이터가 존재 )
            # 변수 -> 함수 호출시 생성이 되고 함수가 종료가 되면 휘발성으로 사라짐
        try:
            # CUD의 경우에는 execute() 쿼리문을 커서에 질의 보낸다. R (select도 여기까지는 공통의 작업)
            # execute( query, () ) -> 호출 가능 
            # execute( query, (1,2,3) ) -> 호출 가능 
            cursor.execute(query, datas)
            # query가 select문이라면? 
            # select * from table // SELECT * FROM table, """   select * from table   """ -> 두가지의 경우 모두 참 
            # 좌측 공백을 제거 , 소문자를 통일 , 시작값이 select와 같은가 startwith()
            if query.lstrip().lower().startswith('select'):
                # 결과값을 받아온다. 
                result = cursor.fetchall()
            else:
                result = "Query OK!"
            return result
        except Exception as e:
            print('query문 execute중 에러')
            print(e)

In [6]:
# class 생성

db1= MyDB(
    host = os.getenv('host'),
    port = int(os.getenv('port')),
    user = os.getenv('user'),
    password = os.getenv('pwd'),
    db = os.getenv('db_name')
)

In [7]:
db1.sql_query('select * from `emp`')

[{'EMPNO': 7369,
  'ENAME': 'SMITH',
  'JOB': 'CLERK',
  'MGR': 7902.0,
  'HIREDATE': '1980-12-17',
  'SAL': 800.0,
  'COMM': 0.0,
  'DEPTNO': 20.0},
 {'EMPNO': 7499,
  'ENAME': 'ALLEN',
  'JOB': 'SALESMAN',
  'MGR': 7698.0,
  'HIREDATE': '1981-02-20',
  'SAL': 1600.0,
  'COMM': 300.0,
  'DEPTNO': 30.0},
 {'EMPNO': 7521,
  'ENAME': 'WARD',
  'JOB': 'SALESMAN',
  'MGR': 7698.0,
  'HIREDATE': '1981-02-22',
  'SAL': 1250.0,
  'COMM': 500.0,
  'DEPTNO': 30.0},
 {'EMPNO': 7566,
  'ENAME': 'JONES',
  'JOB': 'MANAGER',
  'MGR': 7839.0,
  'HIREDATE': '1981-04-02',
  'SAL': 2975.0,
  'COMM': 0.0,
  'DEPTNO': 20.0},
 {'EMPNO': 7654,
  'ENAME': 'MARTIN',
  'JOB': 'SALESMAN',
  'MGR': 7698.0,
  'HIREDATE': '1981-09-28',
  'SAL': 1250.0,
  'COMM': 1400.0,
  'DEPTNO': 30.0},
 {'EMPNO': 7698,
  'ENAME': 'BLAKE',
  'JOB': 'MANAGER',
  'MGR': 7839.0,
  'HIREDATE': '1981-05-01',
  'SAL': 2850.0,
  'COMM': 0.0,
  'DEPTNO': 30.0},
 {'EMPNO': 7782,
  'ENAME': 'CLARK',
  'JOB': 'MANAGER',
  'MGR': 7839.0,
 

In [8]:
# select 확인이 끝났으니 insert update delete 확인 
insert_query = """
    INSERT INTO `user_info`
    VALUES (%s, %s, %s, %s)
"""
select_query = """
    SELECT * FROM `user_info`
"""

data_list = ['test3', '0000', 'lee', 40]
db1.sql_query(insert_query, *data_list)

접속된 서버가 존재함


'Query OK!'

In [9]:
db1.sql_query(select_query)

접속된 서버가 존재함


[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30},
 {'id': 'test3', 'password': '0000', 'name': 'lee', 'age': 40}]

In [10]:
delete_query = """
    DELETE FROM `user_info`
"""

data_list = ['test']

db1.sql_query(delete_query)

접속된 서버가 존재함


'Query OK!'

### 모듈 만들고 가져오기

In [11]:
import db

In [12]:
import importlib
importlib.reload(db)

<module 'db' from 'c:\\Users\\hkssn\\OneDrive\\바탕 화면\\multicam\\20260420\\db.py'>

In [13]:
db3 = db.MyDB()

In [14]:
db3.sql_query(select_query)

[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30}]